<a href="https://colab.research.google.com/github/lorek/MethodsClassDimRed/blob/main/MoCaDR_Demo_CPU_vs_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Methods of classification and dimensionality reduction**


Paweł Lorek  
University of Wrocław

# Comparing running time on CPU vs GPU in Google Colab



 <font face="Rage" size=2  > Updated: 18.02.2026 <font>



This notebook benchmarks training on both CPU and GPU (CUDA), if a GPU is available.

The code defines two devices:

`device_cpu = torch.device("cpu")`  
`device_cuda = torch.device("cuda") if torch.cuda.is_available() else None`

The benchmark is then executed:

- always on **CPU**
- on **GPU (CUDA)** only if it is available

If CUDA is enabled correctly, the notebook will automatically:

- run `benchmark_train(device_cpu)`
- run `benchmark_train(device_cuda)`
- print both execution times
- compute and display the speedup factor



##  How to Enable GPU in Colab

1. Go to **Runtime → Change runtime type**
2. Set **Hardware accelerator** to `GPU`
3. Click **Save**
4. Restart the runtime via **Runtime → Restart runtime**
5. Run all cells again.

If GPU is properly enabled, you should see both CPU and CUDA execution times printed.




In [ ]:
import torch, torch.nn as nn, time


### Check Whether GPU Is Active:



In [ ]:
device_cpu = torch.device("cpu")
device_cuda = torch.device("cuda") if torch.cuda.is_available() else None

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if device_cuda:
    print("GPU name:", torch.cuda.get_device_name(device_cuda))

PyTorch version: 2.9.0+cu128
CUDA available: True
GPU name: Tesla T4


In [ ]:
# Helper to synchronize CUDA for accurate timing
def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

In [ ]:
def benchmark_train(device, steps=400, warmup=50, batch=8192, dim=1024, hidden=2048, classes=10):
    torch.manual_seed(0)
    model = nn.Sequential(
        nn.Linear(dim, hidden),
        nn.ReLU(),
        nn.Linear(hidden, hidden),
        nn.ReLU(),
        nn.Linear(hidden, classes),
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    # Warmup
    for _ in range(warmup):
        x = torch.randn(batch, dim, device=device)
        y = torch.randint(0, classes, (batch,), device=device)
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()

    # Timed
    sync()
    t0 = time.perf_counter()
    for _ in range(steps):
        x = torch.randn(batch, dim, device=device)
        y = torch.randint(0, classes, (batch,), device=device)
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
    sync()
    t1 = time.perf_counter()
    return t1 - t0

def benchmark_matmul(device, iters=300, n=4096):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    for _ in range(10):
        _ = a @ b
    sync()
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = a @ b
    sync()
    t1 = time.perf_counter()
    return t1 - t0

In [ ]:
t_cpu=0

# TRAIN benchmark

In [ ]:
print("\n--- Training benchmark ---")
t_cpu = benchmark_train(device_cpu)
print(f"CPU time:  {t_cpu:.3f} s")


--- Training benchmark ---
CPU time:  996.829 s


In [ ]:
if device_cuda:
    t_gpu = benchmark_train(device_cuda)
    print(f"CUDA time: {t_gpu:.3f} s   (speedup: {t_cpu/t_gpu:.1f}×)")
else:
    print("benchmark_train: cuda unavailable")

CUDA time: 28.028 s   (speedup: 35.6×)


# MATMUL benchmark

In [ ]:
print("\n--- Matrix multiply benchmark ---")
t_cpu = benchmark_matmul(device_cpu)
print(f"CPU time:  {t_cpu:.3f} s")



--- Matrix multiply benchmark ---
CPU time:  320.346 s


In [ ]:
if device_cuda:
    t_gpu = benchmark_matmul(device_cuda)
    print(f"CUDA time: {t_gpu:.3f} s   (speedup: {t_cpu/t_gpu:.1f}×)")
else:
    print("benchmark_matmul: cuda unavailable")

CUDA time: 12.060 s   (speedup: 26.6×)
